Importing Libraries

In [2]:
import pandas as pd

Read Clean Dataset

In [5]:
# Reading a CSV file into a DataFrame
final_df = pd.read_csv('final_df.csv')
final_df.head(5)

,TimeStamp,Compno_x,Compno_y,AverageMaximumTemperature_x,AverageMinimunTemperature_x,AverageMaximumRelativeHumidity_x,AverageMinimumRelativeHumidity_x,TotalRainfall_x,AverageTotalRadiation_x,AverageWindSpeed_x,...,AverageMaximumTemperature_y,AverageMinimunTemperature_y,AverageMaximumRelativeHumidity_y,AverageMinimumRelativeHumidity_y,TotalRainfall_y,AverageTotalRadiation_y,AverageWindSpeed_y,AverageTotalRelativeEvaporation_y,Target_y,distance_km
0,2000-01-01,1,2,28.10,14.77,97.09,39.43,60.4,26.18,1.44,...,32.89,18.73,82.92,23.31,39.0,29.91,1.96,209.35,54.75875,655.071776
1,2000-01-01,1,3,28.10,14.77,97.09,39.43,60.4,26.18,1.44,...,31.25,15.13,93.57,27.33,35.4,28.80,2.49,190.62,53.07375,335.864713
2,2000-01-01,1,4,28.10,14.77,97.09,39.43,60.4,26.18,1.44,...,29.60,15.01,92.16,30.66,69.8,26.97,1.91,170.56,54.58375,220.023323
3,2000-01-01,2,1,32.89,18.73,82.92,23.31,39.0,29.91,1.96,...,28.10,14.77,97.09,39.43,60.4,26.18,1.44,158.57,53.24750,655.071776
4,2000-01-01,2,3,32.89,18.73,82.92,23.31,39.0,29.91,1.96,...,31.25,15.13,93.57,27.33,35.4,28.80,2.49,190.62,53.07375,369.248018


In [17]:
import pandas as pd
import torch
from torch_geometric.data import HeteroData

# Load your data into a pandas DataFrame
# Replace with your actual data loading method

df = final_df

# Debugging step: Print column names
print(df.columns)

# Check if 'Timestamp' is in the columns
if 'Timestamp' in df.columns:
    # Create a heterogeneous data object
    hetero_data = HeteroData()

    # Add nodes and their features
    node_features = {}
    for node in pd.concat([df['Compno_x'], df['Compno_y']]).unique():
        node_df = df[(df['Compno_x'] == node) | (df['Compno_y'] == node)]
        features = node_df.iloc[:, 3:-2].values.mean(axis=0)  # Average features
        node_features[node] = features

    hetero_data['node'].x = torch.tensor(list(node_features.values()), dtype=torch.float)

    # Add edges and their features
    edge_index = []
    edge_attr = []
    for _, row in df.iterrows():
        edge_index.append([row['Compno_x'], row['Compno_y']])
        edge_attr.append(row['distance_km'])

    hetero_data['node', 'connects', 'node'].edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    hetero_data['node', 'connects', 'node'].edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    # Ensure timestamps are handled correctly 
    timestamp_map = {node: idx for idx, node in enumerate(df['Timestamp'].unique())} 
    node_timestamps = [] 
    for node in pd.concat([df['Compno_x'], df['Compno_y']]).unique(): 
        node_df = df[(df['Compno_x'] == node) | (df['Compno_y'] == node)] 
        node_timestamp = timestamp_map[node_df['Timestamp'].iloc[0]] 
        node_timestamps.append(node_timestamp) 
    
    hetero_data['node'].time = torch.tensor(node_timestamps, dtype=torch.float)

    print(hetero_data)
else:
    print("The 'Timestamp' column is missing.")


Index(['TimeStamp', 'Compno_x', 'Compno_y', 'AverageMaximumTemperature_x',
       'AverageMinimunTemperature_x', 'AverageMaximumRelativeHumidity_x',
       'AverageMinimumRelativeHumidity_x', 'TotalRainfall_x',
       'AverageTotalRadiation_x', 'AverageWindSpeed_x',
       'AverageTotalRelativeEvaporation_x', 'Target_x',
       'AverageMaximumTemperature_y', 'AverageMinimunTemperature_y',
       'AverageMaximumRelativeHumidity_y', 'AverageMinimumRelativeHumidity_y',
       'TotalRainfall_y', 'AverageTotalRadiation_y', 'AverageWindSpeed_y',
       'AverageTotalRelativeEvaporation_y', 'Target_y', 'distance_km'],
      dtype='object')
The 'Timestamp' column is missing.


In [11]:
import torch.nn.functional as F
from torch_geometric.nn import GATConv, GraphConv

class TemporalSpatialGNN(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super(TemporalSpatialGNN, self).__init__()
        self.gcn1 = GraphConv(in_channels, 16)
        self.gcn2 = GraphConv(16, out_channels)
        self.temporal_fc = torch.nn.Linear(1, out_channels)

    def forward(self, x, edge_index, edge_attr, time):
        # Spatial GNN
        x = self.gcn1(x, edge_index)
        x = F.relu(x)
        x = self.gcn2(x, edge_index)
        
        # Temporal component
        time_embed = self.temporal_fc(time.float().unsqueeze(1))
        x = x + time_embed
        
        return F.log_softmax(x, dim=1)

model = TemporalSpatialGNN(hetero_data['node'].x.shape[1], 2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)


In [13]:
print(hetero_data)

HeteroData(
  node={ x=[7, 17] },
  (node, connects, node)={
    edge_index=[2, 7364],
    edge_attr=[7364],
  }
)


In [12]:
def train():
    model.train()
    optimizer.zero_grad()
    out = model(hetero_data['node'].x, hetero_data['node', 'connects', 'node'].edge_index, 
                hetero_data['node', 'connects', 'node'].edge_attr, hetero_data['node'].time)
    loss = F.nll_loss(out, torch.tensor([0, 1]))  # Dummy labels, replace with actual labels
    loss.backward()
    optimizer.step()
    return loss.item()

for epoch in range(200):
    loss = train()
    if epoch % 20 == 0:
        print(f'Epoch {epoch}, Loss: {loss:.4f}')


AttributeError: 'NodeStorage' object has no attribute 'time'